# SQL-Analyst Agent — Colab Training Notebook

Reproduces this repo's training runs end to end: clones the repo, installs dependencies, then walks through the SFT arm (`collect_sft_data.py` → `train_sft.py` → held-out eval) and the GRPO arm (`train_grpo.py` → `evaluate_grpo.py`).

**Before running anything:**
1. `Runtime > Change runtime type` → pick a GPU (T4 or better).
2. Add a Colab secret named exactly `WANDB_API_KEY` (key icon, left sidebar) with your Weights & Biases API key.
3. Optional but recommended: add a Colab secret named `HF_TOKEN` (a [Hugging Face token](https://huggingface.co/settings/tokens), read access is enough) - without it, model/dataset downloads are rate-limited harder and can occasionally hit `429 Too Many Requests`.

**One interruption:** after the install cells, you'll be asked to restart the runtime once. This is expected and normal. See the markdown cell where it happens for why. Everything before that point only needs to run once; after restarting, skip straight past it to the cell right after.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected. Runtime > Change runtime type > pick a GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

REPO_URL = "https://github.com/AryamanJaggi/sql-agent-rlvr.git"
REPO_DIR = "/content/sql-agent-rlvr"

# Absolute paths throughout, so this cell is safe to re-run from any cwd -
# a relative `%cd sql-agent-rlvr` re-run from inside an already-nested
# checkout would clone and cd one level deeper every time.
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q --retries 5 --timeout 120 -r requirements.txt
!pip install -q --retries 5 --timeout 120 unsloth vllm trl wandb

In [ ]:
# Unsloth checks the installed vLLM build against this runtime's CUDA
# version and blocks the import if they don't match.
!pip install -q --force-reinstall --no-deps https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl

## Restart runtime now

**Runtime → Restart session**

This is required because Unsloth's import-compatibility check patches Python's import machinery for the rest of the process once it runs, so simply having reinstalled the correct vLLM build above isn't enough within the same running kernel. The block has to be cleared by actually restarting.

**After restarting, continue from the next cell below.** Do not re-run the cells above (clone/install).

In [ ]:
%cd /content/sql-agent-rlvr

import torch
from unsloth import FastLanguageModel
from vllm import SamplingParams
print("Imports OK - vLLM/CUDA mismatch is cleared.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = "/content/drive/MyDrive/sql_agent_rlvr_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results will be saved under:", RESULTS_DIR)

# Higher HF Hub rate limit + fewer 429s on model/dataset downloads.
# transformers/datasets/huggingface_hub all pick this up automatically.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN set from Colab secret.")
except Exception:
    print("No HF_TOKEN secret found - continuing unauthenticated (fine, just slower/stricter rate limits).")

## SFT: collect_sft_data.py → train_sft.py → eval

Runs the untrained prompted baseline over Spider's `train` split (hard/extra difficulty tiers), keeps only its successful trajectories, fine-tunes a LoRA adapter on them, then evaluates that adapter on the `validation` split through the same ReAct harness the baseline was measured with.

Each step below runs a small test first to ensure there are no errors, then the real run.

In [ ]:
!python -m train.collect_sft_data --limit 3 --output {RESULTS_DIR}/sft_data_smoke.jsonl

In [ ]:
!python -m train.collect_sft_data --limit 150 --output {RESULTS_DIR}/sft_data.jsonl

In [ ]:
!python -m train.train_sft --data {RESULTS_DIR}/sft_data_smoke.jsonl --output-dir {RESULTS_DIR}/sft_adapter_smoke --epochs 1

In [ ]:
!python -m train.train_sft --data {RESULTS_DIR}/sft_data.jsonl --output-dir {RESULTS_DIR}/sft_adapter

In [ ]:
!python -m eval.evaluate --policy unsloth --lora-path {RESULTS_DIR}/sft_adapter --split validation --limit 30 --wandb-project sql-agent-rlvr

## GRPO arm: train_grpo.py → evaluate_grpo.py

Trains via TRL's `environment_factory` (native tool-calling), then evaluates with a matching native-tool-calling harness.

**Needs two separate Colab runtimes.** `train_grpo.py` requires `trl>=1.0` for `environment_factory`, which Unsloth doesn't support (every Unsloth release caps `trl<=0.24.0`) - so training runs in a fresh, Unsloth-free runtime, and evaluation switches back to a normal Unsloth runtime (same one used for the SFT arm). The markdown cells below say exactly when to switch.

### Step 1: fresh runtime for train_grpo.py

**Runtime → Disconnect and delete runtime**, then **Runtime → Change runtime type** to pick a GPU again. In that fresh runtime, scroll up and re-run only the **clone/pull cell** and the **Drive-mount cell** from earlier in this notebook - skip the install cells and the Unsloth import-check/restart cells entirely, since those install Unsloth, which this section can't use alongside `environment_factory`. Then run the install cell immediately below, followed by the smoke test and the real training run.

In [ ]:
# Separate from the rest of this notebook: no unsloth here, since
# environment_factory needs trl>=1.0 and Unsloth caps trl<=0.24.0.
# gguf: vllm's config loader imports it unconditionally (GGUF model
# support) even though we never load a GGUF model - not pulled in as
# a hard vllm dependency, so it needs installing explicitly.
!pip install -q --retries 5 --timeout 120 -r requirements.txt
!pip install -q --retries 5 --timeout 120 torch "trl>=1.0,<=1.10.0" peft bitsandbytes transformers vllm gguf wandb
!pip install -q --force-reinstall --no-deps https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl

In [ ]:
!python -m train.train_grpo --limit 5 --num-generations 2 --epochs 1 --output-dir {RESULTS_DIR}/grpo_adapter_smoke

In [ ]:
!python -m train.train_grpo --limit 150 --output-dir {RESULTS_DIR}/grpo_adapter

### Step 2: back to a normal Unsloth runtime for eval

`evaluate_grpo.py` doesn't touch `GRPOTrainer`/`environment_factory` at all - it drives tool-calling manually - so it runs fine in a standard Unsloth runtime, same as the SFT arm. **Runtime → Disconnect and delete runtime**, start a fresh one (or reconnect to the SFT runtime if it's still alive), run the clone/install/restart cells from the top of this notebook (cells 1 through 7), then continue below.

In [ ]:
!python -m eval.evaluate_grpo --lora-path {RESULTS_DIR}/grpo_adapter_smoke --split validation --limit 2 --max-steps 5

In [ ]:
!python -m eval.evaluate_grpo --lora-path {RESULTS_DIR}/grpo_adapter --split validation --limit 30